In [1]:
import pandas as pd 
import numpy as np
import math
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
import joblib

In [2]:
df = pd.read_csv('preprocessed_loan_default.csv')
df

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,LoanDefault
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,1,1,Other,1,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,0,0,Other,1,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,1,1,Auto,0,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,0,0,Business,0,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,0,1,Auto,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255342,8C6S86ESGC,19,37979,210682,541,109,4,14.11,12,0.85,Bachelor's,Full-time,Married,0,0,Other,0,0
255343,98R4KDHNND,32,51953,189899,511,14,2,11.55,24,0.21,High School,Part-time,Divorced,0,0,Home,0,1
255344,XQK1UUUNGP,56,84820,208294,597,70,3,5.29,60,0.50,High School,Self-employed,Married,1,1,Auto,1,0
255345,JAO28CPL4H,42,85109,60575,809,40,1,20.90,48,0.44,High School,Part-time,Single,1,1,Other,0,0


## Training dataset

In [23]:
x = df[['Age','Income','LoanAmount','CreditScore','MonthsEmployed','NumCreditLines','InterestRate','LoanTerm','DTIRatio','HasMortgage','HasDependents']]
y = df['LoanDefault']

x_train, x_test, y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

## Scaling features

In [18]:
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

## Logistic regression

In [19]:
from sklearn.linear_model import LogisticRegression

In [20]:
LR_model = LogisticRegression()
LR_model.fit(x_train_scaled,y_train)

LogisticRegression()

In [21]:
LR_y_pred = LR_model.predict(x_test_scaled)

In [22]:
LR_accuracy = accuracy_score(y_test,LR_y_pred)
print(LR_accuracy)
LR_cm = confusion_matrix(y_test,LR_y_pred)
print("Confusion matrix : ",LR_cm)

0.8845592770636094
Confusion matrix :  [[180198    358]
 [ 23224    498]]


In [15]:
print(classification_report(y_test,LR_y_pred))

              precision    recall  f1-score   support

           0       0.89      1.00      0.94    112847
           1       0.59      0.02      0.05     14827

    accuracy                           0.88    127674
   macro avg       0.74      0.51      0.49    127674
weighted avg       0.85      0.88      0.83    127674



## Saving Logistic Regression Model

In [21]:
joblib.dump(LR_model, "models/logistic_regression.pkl")
joblib.dump(scaler, "models/scaler.pkl")

['models/scaler.pkl']

## Decision Tree

In [36]:
DT_model = DecisionTreeClassifier(max_depth=4,class_weight='balanced')

In [37]:
DT_model.fit(x_train,y_train)

DecisionTreeClassifier(class_weight='balanced', max_depth=4)

In [38]:
DT_y_pred = DT_model.predict(x_test)

In [39]:
DT_accuracy = accuracy_score(y_test,DT_y_pred)
print(DT_accuracy)
DT_cm = confusion_matrix(y_test,DT_y_pred)
print("Confusion matrix : ",DT_cm)

0.6428039945173292
Confusion matrix :  [[28828 16311]
 [ 1931  4000]]


In [40]:
print(classification_report(y_test,DT_y_pred))

              precision    recall  f1-score   support

           0       0.94      0.64      0.76     45139
           1       0.20      0.67      0.30      5931

    accuracy                           0.64     51070
   macro avg       0.57      0.66      0.53     51070
weighted avg       0.85      0.64      0.71     51070



## Saving Decision Tree Model

In [41]:
joblib.dump(DT_model, "models/decision_tree.pkl")

['models/decision_tree.pkl']

## K-Neighbour Model

In [28]:
# Sample 40,000 records strictly from x_train to avoid test data leakage
np.random.seed(42)
knn_indices = np.random.choice(len(x_train), size=40000, replace=False)
x_train_knn = x_train_scaled[knn_indices]
y_train_knn = y_train.iloc[knn_indices].values

# Holdout evaluation set (10,000 clean test records from x_test_scaled)
x_test_scaled_knn = x_test_scaled[:10000]
y_test_knn = y_test.iloc[:10000].values


## Training Model

In [29]:
KNN_model = KNeighborsClassifier(n_neighbors=8,weights='distance')

KNN_model.fit(x_train_scaled_knn, y_train_knn)

KNeighborsClassifier(n_neighbors=8, weights='distance')

In [30]:
KNN_y_pred = KNN_model.predict(x_test_scaled_knn)
# KNN_y_pred

In [31]:
KNN_accuracy = accuracy_score(y_test_knn,KNN_y_pred)
KNN_accuracy

0.8802

In [32]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test_knn, KNN_y_pred))
print(confusion_matrix(y_test_knn, KNN_y_pred))
print(classification_report(y_test_knn, KNN_y_pred))

Accuracy: 0.8802
[[8728  114]
 [1084   74]]
              precision    recall  f1-score   support

           0       0.89      0.99      0.94      8842
           1       0.39      0.06      0.11      1158

    accuracy                           0.88     10000
   macro avg       0.64      0.53      0.52     10000
weighted avg       0.83      0.88      0.84     10000



## Saving KNN model

In [33]:
joblib.dump(KNN_model, "models/k_neighbours.pkl")

['models/k_neighbours.pkl']

## Random Forest

In [34]:
RF_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

## Training Random Forest Model

In [35]:
RF_model.fit(x_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10,
                       min_samples_leaf=5, min_samples_split=10,
                       n_estimators=300, n_jobs=-1, random_state=42)

## Predict using Random Forest

In [36]:
RF_y_pred = RF_model.predict(x_test)

In [37]:
RF_accuracy = accuracy_score(y_test, RF_y_pred)
RF_accuracy

0.72545525748972

## Confusion Matrix and Classification Report

In [38]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, RF_y_pred))

print("\nClassification Report:")
print(classification_report(y_test, RF_y_pred))


Confusion Matrix:
[[33310 11829]
 [ 2192  3739]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.74      0.83     45139
           1       0.24      0.63      0.35      5931

    accuracy                           0.73     51070
   macro avg       0.59      0.68      0.59     51070
weighted avg       0.86      0.73      0.77     51070



## Saving Random Forest Model

In [39]:
joblib.dump(RF_model, "models/random_forest.pkl")

['models/random_forest.pkl']

## All model comparison

In [42]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

models = {
    "Logistic Regression": (LR_model, LR_y_pred, x_test_scaled, y_test),
    "Decision Tree": (DT_model, DT_y_pred, x_test, y_test),
    "Random Forest": (RF_model, RF_y_pred, x_test, y_test),
    "K-Nearest Neighbours": (KNN_model, KNN_y_pred, x_test_scaled_knn, y_test_knn)
}

for name, (model, y_pred, X_test_model, y_test_model) in models.items():
    y_prob = model.predict_proba(X_test_model)[:, 1]

    print("\n", name)
    print("Accuracy :", accuracy_score(y_test_model, y_pred))
    print("Precision:", precision_score(y_test_model, y_pred))
    print("Recall   :", recall_score(y_test_model, y_pred))
    print("F1 Score :", f1_score(y_test_model, y_pred))
    print("ROC-AUC  :", roc_auc_score(y_test_model, y_prob))



 Logistic Regression
Accuracy : 0.8843743880947719
Precision: 0.5555555555555556
Recall   : 0.02191873208565166
F1 Score : 0.042173560421735604
ROC-AUC  : 0.7440373887871536

 Decision Tree
Accuracy : 0.6702565106716272
Precision: 0.19133042838549036
Recall   : 0.5700556398583713
F1 Score : 0.2865011439708499
ROC-AUC  : 0.6616391753651301

 Random Forest
Accuracy : 0.72545525748972
Precision: 0.240172147995889
Recall   : 0.6304164559096274
F1 Score : 0.3478301316340295
ROC-AUC  : 0.7480839799702381

 K-Nearest Neighbours
Accuracy : 0.8802
Precision: 0.39361702127659576
Recall   : 0.06390328151986183
F1 Score : 0.1099554234769688
ROC-AUC  : 0.6560877410724993
